# 052 — Visualising Filters and Feature Maps

A CNN does not have to be a black box. The geometric reason deep feature maps
look abstract: a first-layer neuron sees **3x3 pixels, 0.02%** of a 224x224
image; a last-layer neuron sees **196x196, 76.6%**.

Computed from the architecture; the picture-plotting part needs TensorFlow and
is left as an exercise.

| Part | What we check |
|---|---|
| A | filters against feature maps - what each one belongs to |
| B | receptive field growth through all 13 VGG16 layers |
| C | 0.02% to 76.6% of the image |
| D | a hand-built filter responding to an edge |

## Part A — Two different things

A **filter** is weights: it belongs to the model, and you can plot it before
the model has ever seen your image. A **feature map** is an activation: it
belongs to the model *and* one particular image.

In [ ]:
rows = [("Filter", "the weights of one kernel", "3x3x3 for a first-layer VGG16 filter",
         "the model"),
        ("Feature map", "what that filter found", "224x224 per filter, per image",
         "the model and one image")]
for name, what, shape, belongs in rows:
    print(f"{name:<12} {what:<28} {shape:<38} belongs to: {belongs}")
print()
print("A model has as many filters as it was built with, and as many feature")
print("maps as filters x images you push through it.")
assert rows[0][3] != rows[1][3]

## Part B — How far a neuron can see

In [ ]:
BLOCKS = [(64, 2), (128, 2), (256, 3), (512, 3), (512, 3)]
INPUT = 224

rf, jump, trace = 1, 1, []
for block, (filters, repeats) in enumerate(BLOCKS, 1):
    for r in range(repeats):
        rf += (3 - 1) * jump                 # 3x3 convolution, stride 1
        trace.append((f"block{block}_conv{r + 1}", rf, rf * rf / (INPUT * INPUT)))
    rf += (2 - 1) * jump                     # 2x2 max pool, stride 2
    jump *= 2                                # the pool doubles the jump

for name, size, share in trace:
    print(f"  {name:<14} {size:>3}x{size:<3} px   {min(share, 1):>6.2%} of the image")
assert len(trace) == 13
assert trace[0][1] == 3 and trace[-1][1] == 196

## Part C — The exchange rate

In [ ]:
first, last = trace[0], trace[-1]
print(f"{first[0]:<14} {first[1]}x{first[1]} px  = {first[2]:.2%} of the image")
print(f"{last[0]:<14} {last[1]}x{last[1]} px = {last[2]:.2%} of the image")
print(f"growth: {last[1] / first[1]:.0f}x in side length, "
      f"{last[2] / first[2]:,.0f}x in area")
assert round(first[2] * 100, 2) == 0.02
assert round(last[2] * 100, 1) == 76.6
print()
print("Every pooling stage trades spatial detail for semantic reach. A deep")
print("neuron looks abstract because it is literally looking at most of the")
print("photograph - there is no mystery to explain.")

## Part D — One filter, by hand

An edge detector is three numbers. Run it over a synthetic image and the
response is the feature map.

In [ ]:
import numpy as np

image = np.zeros((9, 9))
image[:, 4:] = 1.0                    # a vertical edge down the middle
vertical = np.array([[-1, 0, 1]] * 3, dtype=float)      # 3x3 edge detector

def convolve(img, kernel):
    kh, kw = kernel.shape
    out = np.zeros((img.shape[0] - kh + 1, img.shape[1] - kw + 1))
    for y in range(out.shape[0]):
        for x in range(out.shape[1]):
            out[y, x] = (img[y:y + kh, x:x + kw] * kernel).sum()
    return out

vmap = convolve(image, vertical)
hmap = convolve(image, vertical.T)    # the same filter, rotated
print("vertical filter, response per column:", vmap[0].round(2))
print("horizontal filter, response per column:", hmap[0].round(2))
assert vmap.max() == 3.0 and abs(hmap).max() == 0.0
print()
print("The vertical filter fires on the edge; the horizontal one sees nothing.")
print("That is all a feature map is: one filter's answer about one image.")

## What this notebook established

- A filter is weights (model only); a feature map is an activation (model plus
  one image).
- A VGG16 neuron's receptive field grows **3x3 → 196x196** across 13
  convolutional layers, from **0.02%** to **76.6%** of a 224x224 image.
- Growth comes from the pooling stages doubling the jump, not from the
  convolutions themselves.
- Early layers respond to edges because an edge is all a 3x3 window can hold.

## Exercises

1. Plot `trace` with matplotlib. Where does the curve bend, and why there?
2. Change the pooling to stride 1 and recompute. How deep would the network
   need to be to see the whole image?
3. Write a diagonal edge filter and confirm it fires on a diagonal image and
   not on the vertical one.
4. With TensorFlow available: load VGG16, plot `block1_conv1`'s 64 filters, and
   tap `block5_conv3` on one photograph.